In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# =============================================================================
# File name: paris_flood_dataset_helper.py
# Author: M51-ULS-1b
# Date created: 2026-03-02
# Version = "1.0"
# License =  "CC0"
# Location = "43° 16′ 41″ N, 2° 39′ 30″ E"
# =============================================================================
""" Helper functions for the Paris flood dataset exploratory data analysis 
and visualization notebook"""
# =============================================================================

from numpy import ndarray
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Union, Sequence, Iterable, Literal
import warnings

import math
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from joypy import joyplot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# ---------------------------------------------------------------------------
# Parameters and setup

# Type aliases
NDArray = np.ndarray

# Display
pd.set_option("display.float_format", lambda x: "%.2f" % x)

plt.rcParams["axes.formatter.useoffset"] = False
plt.rcParams["axes.formatter.use_mathtext"] = False
plt.rcParams["axes.formatter.limits"] = (-7, 7)
plt.rcParams["figure.dpi"] = 72

# Constants
FIG_SIZES = {
    "single": (8, 6),
    "missing": (10, 8),
    "grid": (12, 10),
    "joyplot": (6, 7),
    "lineplots": (10, 4)
}

Z_THRESH = 5.1
MIN_STD = 1e-6

DAY_RANGE = np.arange(1, 367)
MONTH_TICKS = {
    "positions": [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335],
    "labels": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
}

# https://matplotlib.org/stable/gallery/color/named_colors.html
# https://matplotlib.org/stable/gallery/color/colormap_reference.html
STYLES = {
    "classic": {
        "comment": "Inspired by the classic joyplot",
        "title": "125 YEARS OF DATA",
        "years": [],
        "cmap": None,
        "norm_func": None,
        "linewidth": 0.6,
        "highlight_width": None,
        "bgcolor": "k",
        "default_color": "w",
        "flood_metric": lambda d: d.groupby("Year")["water_level_mm"].max()
    },
    "calm": {
        "comment": "Steadiest years. Calm style.",
        "title": "C A L M ",
        "years": ["1921", "1949", "1971", "2009", "1954"],
        "cmap": plt.get_cmap("winter"),
        "norm_func": lambda mn, mx: mcolors.Normalize(vmin=mn, vmax=mx),
        "linewidth": 0.6,
        "highlight_width": 1.3,
        "bgcolor": "grey",
        "default_color": "bisque",
        "flood_metric": lambda d: (
            d.groupby("Year")["water_level_mm"]
            .apply(lambda x: x[x != 0].std())
        )
    },
    "excess": {
        "comment": "Years with highest divergence, typical of flood years.",
        "title": "E X C E S S ",
        "years": ["1910", "1919", "2018", "1982", "1970"],
        "cmap": plt.get_cmap("copper_r"),
        "norm_func": lambda vmin, vmax: mcolors.PowerNorm(gamma=0.6,
                                                          vmin=vmin,
                                                          vmax=vmax),
        "linewidth": 0.6,
        "highlight_width": 1.3,
        "bgcolor": "grey",
        "default_color": "bisque",
        "flood_metric": lambda daily: (
            daily.groupby("Year")["water_level_mm"]
            .apply(lambda x: x[x != 0].max())
        )
    },
    "lutetia": {
        "comment": "Excess inspired by the colors of the Paris herald, but not the original ones (not harmonious in this case).",
        "title": "F L U C T U A T   N E C   M E R G I T U R",
        "years": ["1910", "1919", "2018", "1982", "1970"],
        "cmap": "#c0392b",  # d90000
        "norm_func": None,
        "linewidth": 0.6,
        "highlight_width": 1.3,
        "bgcolor": "#2980b9",  # 23005d
        "default_color": "azure",
        "flood_metric": lambda daily: (
            daily.groupby("Year")["water_level_mm"]
            .apply(lambda x: x[x != 0].max()))
    }
}


# ---------------------------------------------------------------------------
# Utility functions

def find_missing_periods(df: pd.DataFrame, date_col: str = "record_date", group_col: str = "location_code"):
    """
    Identify missing time periods (years, months, and days) in a grouped dataset.

    This function analyzes a DataFrame grouped by a specified column and identifies
    gaps in the date coverage for each group. It compares the dates present in the
    data against all possible dates within the observed date range (from earliest
    to latest record) and returns the missing periods at three levels of granularity.

    Args:
        df : pd.DataFrame
            The input DataFrame containing date and grouping information.
        date_col : str, optional
            Name of the column containing date values. Defaults to "record_date".
            Values are coerced to datetime and normalized to date objects.
        group_col : str, optional
            Name of the column to group by (e.g., location or entity identifier).
            Defaults to "location_code".

    Returns:
        dict
            A dictionary with three keys, each containing a nested dictionary mapping
            group identifiers to lists of missing periods:
            - "missing_years": List of years (int) with no records for each group.
            - "missing_months": List of months (str, format "YYYY-MM") with no records.
            - "missing_days": List of days (str, format "YYYY-MM-DD") with no records.

    Notes:
    - Invalid dates in `date_col` are coerced to NaT and excluded from analysis.
    - Groups with no valid dates return empty lists for all period types.
    - Missing periods are only identified within the range [min_date, max_date]
      observed for each group; periods before or after this range are not flagged.
    """
    # make sure datetime (keep only date part)
    df = df.copy()
    df[date_col] = pd.to_datetime(
        df[date_col], errors="coerce").dt.normalize().dt.date

    results = {"missing_years": {}, "missing_months": {}, "missing_days": {}}

    for key, grp in df.groupby(group_col):
        dates = grp[date_col].dropna().unique()
        if len(dates) == 0:
            results["missing_years"][key] = []
            results["missing_months"][key] = []
            results["missing_days"][key] = []
            continue

        dates = pd.to_datetime(pd.Series(dates))
        start, end = dates.min().date(), dates.max().date()

        # Years
        present_years = set(dates.dt.year.astype(int).tolist())
        all_years = set(range(start.year, end.year + 1))
        results["missing_years"][key] = sorted(all_years - present_years)

        # Months (YYYY-MM)
        present_months = set(dates.dt.to_period("M").astype(str).tolist())
        all_months = set(pd.period_range(start=pd.Period(start, freq="M"),
                                         end=pd.Period(end, freq="M"),
                                         freq="M").astype(str).tolist())
        results["missing_months"][key] = sorted(all_months - present_months)

        # Days (YYYY-MM-DD)
        present_days = set(dates.dt.date.astype(str).tolist())
        all_days = set(pd.date_range(start=start, end=end,
                       freq="D").date.astype(str).tolist())
        results["missing_days"][key] = sorted(all_days - present_days)

    return results


def flood_alert_stats(df: pd.DataFrame,
                      date_col: str = "record_date",
                      value_col: str = "water_level_mm",
                      alert_col: str = "flood_alert",
                      group_col: str = "location_code"):
    """
    Compute flood alert statistics grouped by location, including duration analysis.

    This function analyzes flood alert records in a DataFrame and calculates summary
    statistics for water level values during alert periods. It identifies continuous
    alert date ranges and determines the longest consecutive alert duration for each
    group. An overall summary of the longest alert period(s) across all groups is
    printed to the console.

    Args:
        df : pd.DataFrame
            The input DataFrame containing flood alert and water level data.
        date_col : str, optional
            Name of the column containing date values. Defaults to "record_date".
            Values are coerced to datetime and converted to date objects.
        value_col : str, optional
            Name of the column containing water level measurements. Defaults to "water_level_mm".
            Non-numeric values are coerced to NaN and excluded from statistics.
        alert_col : str, optional
            Name of the boolean column indicating flood alert status. Defaults to "flood_alert".
            Values are coerced to boolean type.
        group_col : str, optional
            Name of the column to group by (e.g., location identifier). Defaults to "location_code".

    Returns:
        dict
            A dictionary mapping group identifiers to their alert statistics. Each group
            contains the following keys:
            - "count": Number of alert records (int).
            - "mean": Mean water level during alerts (float or None if no data).
            - "median": Median water level during alerts (float or None if no data).
            - "min": Minimum water level during alerts (float or None if no data).
            - "max": Maximum water level during alerts (float or None if no data).
            - "std": Standard deviation of water level during alerts (float or None if no data).
            - "date_ranges": List of tuples (start_date, end_date) in ISO format representing
            continuous alert periods.
            - "longest_alert_days": Length of the longest consecutive alert period in days (int).

    Notes:
    - Only records where `alert_col` is True are included in the analysis.
    - Consecutive alert dates are identified based on 1-day differences; gaps indicate
      separate alert periods.
    - If no alerts are found in the entire dataset, "No alerts found." is printed.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce").dt.date
    df[alert_col] = df[alert_col].astype(bool)

    results = {}
    overall_longest = 0
    # group -> list of ranges with that length
    overall_ranges = defaultdict(list)

    for key, grp in df.groupby(group_col):
        g = grp[grp[alert_col]].sort_values(
            by=date_col).dropna(subset=[date_col])
        if g.empty:
            results[key] = {
                "count": 0,
                "mean": None,
                "median": None,
                "min": None,
                "max": None,
                "std": None,
                "date_ranges": [],
                "longest_alert_days": 0
            }
            continue

        vals = pd.to_numeric(g[value_col], errors="coerce").dropna()
        stats = {
            "count": int(len(vals)),
            "mean": float(vals.mean()) if not vals.empty else None,
            "median": float(vals.median()) if not vals.empty else None,
            "min": float(vals.min()) if not vals.empty else None,
            "max": float(vals.max()) if not vals.empty else None,
            "std": float(vals.std(ddof=0)) if len(vals) > 0 else None
        }

        dates = pd.Series(sorted(g[date_col].unique()))
        dates_dt = pd.to_datetime(dates).reset_index(drop=True)
        ranges = []
        longest = 0
        if not dates_dt.empty:
            diffs = dates_dt.diff().dt.days.fillna(0).astype(int)
            start = dates_dt.iloc[0].date()
            prev = start
            longest = 1
            for d, diff in zip(dates_dt.iloc[1:], diffs.iloc[1:]):
                d_date = d.date()
                if diff == 1:
                    prev = d_date
                    current_len = (prev - start).days + 1
                    if current_len > longest:
                        longest = current_len
                    continue
                else:
                    ranges.append((start.isoformat(), prev.isoformat()))
                    # record if this run matches/sets overall_longest later
                    start = d_date
                    prev = d_date
            ranges.append((start.isoformat(), prev.isoformat()))
            if longest == 0 and len(dates_dt) > 0:
                longest = 1

        stats["date_ranges"] = ranges
        stats["longest_alert_days"] = int(longest)
        results[key] = stats

        # track overall longest and corresponding ranges
        if stats["longest_alert_days"] > overall_longest:
            overall_longest = stats["longest_alert_days"]
            overall_ranges.clear()
            overall_ranges[key].extend(r for r in stats["date_ranges"]
                                       if (pd.to_datetime(r[1]).date() - pd.to_datetime(r[0]).date()).days + 1 == overall_longest)
        elif stats["longest_alert_days"] == overall_longest and overall_longest > 0:
            overall_ranges[key].extend(r for r in stats["date_ranges"]
                                       if (pd.to_datetime(r[1]).date() - pd.to_datetime(r[0]).date()).days + 1 == overall_longest)

    if overall_longest == 0:
        print("No alerts found.")
    else:
        print(f"Overall longest alert: {overall_longest} days")
        for grp, ranges in overall_ranges.items():
            for r in ranges:
                print(f"- Group {grp}: {r[0]} to {r[1]}")

    return results


def detect_outliers(
    df: pd.DataFrame,
    water_lev_col: str = "water_level_mm",
    z_thresh: float = Z_THRESH,
    min_std: float = MIN_STD,
    dt_col: str = "record_date",
    group_by: Literal["location", "location_month"] = "location"
) -> pd.DataFrame:
    """
    Detect statistical outliers in water level data using Z-score thresholding.

    This function identifies suspicious water level measurements by computing Z-scores
    within specified groupings and flagging values that exceed a threshold. Outliers
    are only detected in groups with sufficient variance and sample size to ensure
    statistical validity.

    Args:
        df : pd.DataFrame
            The input DataFrame containing water level measurements and grouping information.
        water_lev_col : str, optional
            Name of the column containing water level values. Defaults to "water_level_mm".
        z_thresh : float, optional
            Absolute Z-score threshold for outlier detection. Values with |Z-score| > z_thresh
            are flagged as outliers. Defaults to Z_THRESH (module-level constant).
        min_std : float, optional
            Minimum standard deviation required for a group to be evaluated for outliers.
            Groups with std < min_std are not screened. Defaults to MIN_STD (module-level constant).
        dt_col : str, optional
            Name of the datetime column. Defaults to "record_date".
            Used for grouping when group_by="location_month" and for reporting.
        group_by : {"location", "location_month"}, optional
            Grouping strategy for computing Z-scores:
            - "location": Compute statistics per location_code (default).
            - "location_month": Compute statistics per location_code and year-month combination,
            enabling seasonal or monthly anomaly detection.

    Returns:
        pd.DataFrame
            A subset of the input DataFrame containing only rows identified as outliers.
            Includes all original columns plus computed columns:
            - "year_month": Period object (if group_by="location_month").
            - "mean": Group mean water level.
            - "std": Group standard deviation.
            - "n": Group sample size.
            - "z_score": Computed Z-score for the record.
    """
    df_copy = df.copy()

    if group_by == "location_month":
        df_copy[dt_col] = pd.to_datetime(df_copy[dt_col], errors="coerce")
        df_copy = df_copy.dropna(subset=[dt_col])
        df_copy["year_month"] = df_copy[dt_col].dt.to_period("M")
        group_cols = ["location_code", "year_month"]
    else:
        group_cols = ["location_code"]

    stats_df = df_copy.groupby(group_cols)[water_lev_col].agg(
        n="size", mean="mean", std="std"
    ).reset_index()

    merged = df_copy.merge(stats_df, on=group_cols)
    merged["std"] = merged["std"].replace(0, np.nan)  # match original behavior
    merged["z_score"] = (merged[water_lev_col] -
                         merged["mean"]) / merged["std"]

    mask = (merged["n"] > 1) & (merged["std"] >= min_std) & (
        merged["z_score"].abs() > z_thresh)
    outliers = merged[mask].copy()

    if dt_col in outliers.columns:
        try:
            print(
                f'{outliers[dt_col].nunique()} days containing {len(outliers)} outliers')
        except Exception:
            print(f'{len(outliers)} outliers found')
    else:
        print(f'{len(outliers)} outliers found')

    return outliers


def match_z_outliers_to_alert_ranges(
    z_score_outliers: pd.DataFrame,
    alert_stats: Dict[str, dict],
    group_col: str = "location_code",
    z_date_col: str = "record_date"
) -> pd.DataFrame:
    """
    Filter Z-score outliers to include only those occurring during flood alert periods.

    This function cross-references statistical outliers (detected via Z-score analysis)
    with known flood alert date ranges. Only outliers whose dates fall within active
    alert periods are retained.

    Args:
        z_score_outliers : pd.DataFrame
            DataFrame of outliers detected via Z-score thresholding, typically from
            detect_outliers(). Must contain a date column and optionally a grouping column.
        alert_stats : Dict[str, dict]
            Dictionary mapping group identifiers to alert statistics, typically from
            flood_alert_stats(). Each group's dictionary should contain a "date_ranges" key
            with a list of tuples (start_date, end_date) in ISO format (YYYY-MM-DD).
        group_col : str, optional
            Name of the grouping column in z_score_outliers (e.g., location identifier).
            Defaults to "location_code". If the column does not exist, all outliers are
            evaluated as a single group.
        z_date_col : str, optional
            Name of the date column in z_score_outliers. Defaults to "record_date".
            Values are coerced to datetime and normalized to midnight for interval matching.

    Returns:
        pd.DataFrame
            A filtered DataFrame containing only outliers whose dates overlap with alert
            date ranges for their respective groups. The result is sorted by original index
            and reset to a clean integer index. All original columns are preserved.

    Notes:
    - Dates are normalized to midnight (00:00:00) before interval matching.
    - Alert date ranges are treated as closed intervals [start_date, end_date], so
      outliers on start or end dates are included.
    - Groups present in z_score_outliers but absent in alert_stats are skipped
      (no outliers retained for those groups).
    """
    z = z_score_outliers.copy()
    # normalize to Timestamp midnight
    z[z_date_col] = pd.to_datetime(
        z[z_date_col], errors="coerce").dt.normalize()

    if group_col in z.columns:
        groups = z[group_col].unique()
    else:
        groups = [None]

    keep_idx = []

    for grp in groups:
        key = grp
        stats = alert_stats.get(key, {}) if key is not None else {}
        ranges: List[Tuple[str, str]] = stats.get("date_ranges", [])
        if not ranges:
            continue

        # build IntervalIndex of Timestamps
        starts_ts = pd.to_datetime([s for s, _ in ranges])
        ends_ts = pd.to_datetime([e for _, e in ranges])
        intervals = pd.IntervalIndex.from_arrays(
            starts_ts, ends_ts, closed="both")

        if key is None:
            candidate_idx = z.index
        else:
            candidate_idx = z.index[z[group_col] == key]

        if candidate_idx.empty:
            continue

        dates = pd.to_datetime(z.loc[candidate_idx, z_date_col]).dt.normalize()

        # vectorized membership test
        def in_any_interval(ts):
            if pd.isna(ts):
                return False
            return intervals.contains(ts).any()

        mask = dates.apply(in_any_interval)
        keep_idx.extend(mask[mask].index.tolist())

    result = z.loc[sorted(set(keep_idx))].reset_index(drop=True)
    return result


def prepare_data(
    df: pd.DataFrame,
    dt_col: str = "record_date",
    water_lev_col: str = "water_level_mm",
) -> Tuple[ndarray, List[str], pd.DataFrame]:
    """
    Prepare time series data for analysis by:
    1. Parsing datetime column to midnight timestamps
    2. Creating a full day index from Jan 1 of min year to Dec 31 of max year
    3. Aggregating values by day (keeping maximas)
    4. Merging into full index, keeping NaNs for missing days
    5. Pivoting into (years x days) array

    Args:
        df (DataFrame): Input DataFrame containing time series data.
        dt_col (str, optional): Name of the datetime column. Defaults to "record_date".
        water_lev_col (str, optional): Name of the column with numerical values. Defaults to "water_level_mm".

    Returns:
        Tuple containing:
        - ndarray: 2D array of daily maximum values (years x days)
        - List[str]: List of year labels
        - DataFrame: Processed daily data
    """
    # 1) copy & normalize to midnight
    df_copy = df.copy()
    df_copy[dt_col] = pd.to_datetime(df_copy[dt_col]).dt.normalize()
    first_year = df_copy[dt_col].dt.year.min()     # extract year span
    last_year = df_copy[dt_col].dt.year.max()
    # 2) full daily index from Jan 1 first_year to Dec 31 last_year
    full = pd.DataFrame({
        dt_col: pd.date_range(
            start=pd.Timestamp(first_year, 1, 1),
            end=pd.Timestamp(last_year, 12, 31),
            freq="D",
        )
    })
    # 3) daily max of water_lev_col
    daily_max = (
        df_copy
        .groupby(dt_col, as_index=False)[water_lev_col]
        .max()
    )
    daily_max[dt_col] = daily_max[dt_col].dt.tz_localize(None)
    # 4) merge & keep NaNs
    daily_metrics_df = (
        full
        .merge(daily_max, on=dt_col, how="left")
        .assign(
            # water_level_mm will now contain NaN wherever no original data existed
            water_level_mm=lambda d: d[water_lev_col].fillna(0),
            # missing values flag - True if exactly 0
            is_zero=lambda d: d[water_lev_col] == 0,
            Year=lambda d: d[dt_col].dt.year,
            Day=lambda d: d[dt_col].dt.dayofyear,
        )
    )
    # 5) pivot: years × day-of-year, leave NaNs in pivot
    pivot = (
        daily_metrics_df
        .pivot(index="Year", columns="Day", values=water_lev_col)
        .reindex(columns=DAY_RANGE)     # ensure columns 1…366
        .sort_index(ascending=False)    # newest year first
    )
    year_day_matrix = pivot.to_numpy()     # shape (n_years, 366), with NaNs
    yr_labels = pivot.index.astype(str).tolist()

    return year_day_matrix, yr_labels, daily_metrics_df


# ---------------------------------------------------------------------------
# Plot functions

def plot_frequency_histogram(df: pd.DataFrame) -> None:
    """
    Plot histograms for each numeric column in the DataFrame.

    Args:
    df (DataFrame): Input DataFrame.

    Returns:
        None. Displays visualization.

    Behavior:
    - Automatically selects numeric columns (after coercing where reasonable).
    - Skips columns with no valid numeric data.
    - Arranges subplots in a grid and shows each column's histogram.
    """
    df = df.copy()
    num_df = df.select_dtypes(include=['number']).dropna(axis=1, how='all')
    if num_df.shape[1] == 0:
        print("No numeric columns to plot.")
        return

    # Set up grid: sensible shape
    n = num_df.shape[1]
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = axes.flatten() if isinstance(axes, (list, np.ndarray)) else [axes]

    for ax in axes[n:]:
        ax.set_visible(False)

    for i, (col_name, series) in enumerate(num_df.items()):
        ax = axes[i]
        series = series.dropna()
        if series.empty:
            ax.set_visible(False)
            continue
        ax.hist(series, bins=30, color='#2b8cbe', edgecolor='k', alpha=0.8)
        ax.set_title(f"Freq. distribution: {col_name}")
        ax.set_ylabel("Frequency")
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        ax.grid(False, axis="x")

    plt.tight_layout()
    plt.show()


def plot_missing_values(df: pd.DataFrame) -> None:
    """
    Analyze and visualize missing values in a dataset similar to the sample provided.
    - Coerces obvious numeric-like columns to numeric before analysis.
    - Prints per-column missing counts and a small sample of rows with any missing values.
    - Shows a single heatmap of the missing-data pattern (not one per column).
    - Reports pairwise missingness correlations and a short interpretation.

    Args:
        df (DataFrame): Input DataFrame to analyze for missing values.

    Returns:
        None. Prints statistics and displays visualization.

    Notes:
        #TODO - Known bug: on large datasets, missing values might not display properly (lines too thin).
    """
    df = df.copy()
    # Coerce specified numeric columns to numeric (non-convertible values will be set to NaN)
    for col in df.select_dtypes(include=['number']).columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Standard missing-value counts
    missing_counts = df.isnull().sum()
    print("Missing values per column:")
    print(missing_counts[missing_counts > 0]
          if missing_counts.any() else missing_counts)

    if not df.isnull().values.any():
        print("No missing data found. Plot not necessary.")
        return

    rows_with_missing = df[df.isnull().any(axis=1)]
    print(
        f"\nRows with missing values (showing up to 10 rows, total {len(rows_with_missing)}):")
    print(rows_with_missing.head(10))

    # Single heatmap for the whole dataframe's missing pattern
    plt.figure(figsize=FIG_SIZES["missing"])
    sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False)
    plt.title("Missing-data pattern (columns on x-axis)")
    plt.xlabel("Columns")
    plt.show()

    # Pairwise missingness correlation matrix (useful to detect related missingness)
    isnull_df = df.isnull().astype(int)
    if isnull_df.shape[1] >= 2:
        corr = isnull_df.corr()
        # Report strong correlations (> 0.5) of missingness between columns
        strong_pairs = []
        for i, col in enumerate(corr.columns):
            for j in range(i + 1, len(corr.columns)):
                other = corr.columns[j]
                val = corr.at[col, other]
                if pd.notna(val) and abs(val) > 0.5:
                    strong_pairs.append((col, other, val))

        if strong_pairs:
            print("\nStrong missingness correlations found (abs(corr) > 0.5):")
            for a, b, v in strong_pairs:
                relation = "positive" if v > 0 else "negative"
                print(
                    f" - {a} <-> {b}: {v:.2f} ({relation}) — may indicate related collection issues or MAR")
        else:
            print(
                "\nNo strong pairwise missingness correlations detected (abs(corr) <= 0.5).")

    # Simple guidance based on overall missingness fraction per column
    frac_missing = (missing_counts / len(df)).sort_values(ascending=False)
    print("\nFraction missing per column (top 10):")
    print(frac_missing.head(10))

    print("\nRecommendations:")
    print("- If a column has >30% missing, consider dropping or obtaining more data.")
    print("- If missingness correlates with other fields, consider imputation conditional on those fields.")
    print("- For numeric columns with low missingness, consider median imputation; for categorical, mode.")


def plot_distribution(
    df: pd.DataFrame,
    cols: Optional[Iterable[str]] = None,
    grid: bool = False
) -> None:
    """
    Plot distribution visualizations for numeric columns.

    Generates histograms, kernel density estimates, Q-Q plots, and boxplots 
    for each specified numeric column in the DataFrame.

    Args:
        df (DataFrame): Input DataFrame.
        cols (Optional[Iterable[str]]): Column name or iterable of column names to process.
            If None, all numeric columns (int/float) are used.
        grid (bool, optional): Whether to display plots in a grid layout. Defaults to False.

    Returns:
        None. Displays distribution plots and prints summary statistics.
    """
    # Resolve columns: if None, use all numeric; else validate and keep only numeric ones
    if cols is None:
        num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    else:
        # allow single string or iterable
        if isinstance(cols, str):
            cols_iter = [cols]
        else:
            cols_iter = list(cols)
        # keep only columns that exist in df
        cols_exist = [c for c in cols_iter if c in df.columns]
        if not cols_exist:
            raise ValueError(
                "No valid columns found in DataFrame for the provided `cols`.")
        # filter to numeric dtype columns
        num_cols = [
            c for c in cols_exist if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            raise ValueError(
                "Specified columns do not contain any numeric columns.")

    for col in num_cols:
        data = df[col].dropna()
        if data.empty:
            print(f"Skipping {col}: no non-null values.")
            continue

        if grid:
            fig, axes = plt.subplots(2, 2, figsize=FIG_SIZES["grid"])
            axes = axes.flatten()
        else:
            axes = [plt.figure(figsize=FIG_SIZES["single"]).gca()
                    for _ in range(4)]

        # Histogram
        axes[0].hist(data, bins=50, color="#7ca1cc", edgecolor="black")
        axes[0].set_title(f"Histogram: {col}")

        # Density
        sns.kdeplot(data, fill=True, ax=axes[1], color="#2a9d8f")
        axes[1].set_title(f"Density: {col}")

        # Q-Q
        stats.probplot(data, plot=axes[2])
        # style Q-Q points/fit if available
        try:
            lines = axes[2].get_lines()
            if len(lines) >= 2:
                pts, fit = lines[:2]
                pts.set_markerfacecolor("#7ca1cc")
                pts.set_markeredgecolor("#7ca1cc")
                fit.set_color("#a50b5e")
                fit.set_linestyle("--")
        except Exception:
            pass
        axes[2].set_title(f"Q-Q plot: {col}")

        # Boxplot
        axes[3].boxplot(data, vert=True, patch_artist=True,
                        boxprops=dict(facecolor="#e9c46a"))
        axes[3].set_title(f"Boxplot: {col}")

        plt.tight_layout()
        plt.show()

        # Summary stats
        summary = data.agg(["mean", "median", "std", "skew", "kurtosis"])
        mode = data.mode().iloc[0] if not data.mode().empty else np.nan
        print(f"{col} — mean: {summary['mean']}, med.: {summary['median']}, mode: {mode}, "
              f"std: {summary['std']}, skew.: {summary['skew']}, kurt.: {summary['kurtosis']}")


def plot_unknown_pleasures(df: pd.DataFrame, dt_col: str = "record_date", water_lev_col: str = "water_level_mm") -> None:
    """
    Plot a classic joyplot of daily maximum water levels by year.

    Args:
        df (pd.DataFrame): Input DataFrame containing time series data.
        dt_col (str, optional): Name of the datetime column. Defaults to "record_date".
        water_lev_col (str, optional): Name of the column with numerical values. Defaults to "water_level_mm".

    Returns:
        None. Displays a joyplot of daily maximum water levels.
    """
    df_copy = df.copy()
    df_copy[dt_col] = pd.to_datetime(df_copy[dt_col])
    daily_metrics_df = (
        df_copy
        .assign(day=lambda d: d[dt_col].dt.normalize())
        .groupby("day", as_index=False)[water_lev_col]
        .max()
        .rename(columns={"day": "date"})
    )

    daily_metrics_df["Year"] = daily_metrics_df["date"].dt.year
    vmin, vmax = daily_metrics_df[water_lev_col].min(
    ), daily_metrics_df[water_lev_col].max()

    grouped = list(daily_metrics_df.groupby("Year", observed=True))

    groups = [g[water_lev_col].values for _, g in grouped]
    labels = [str(year) for year, _ in grouped]

    fig, axes = joyplot(
        groups,
        labels=labels,
        kind="counts",
        bins=80,
        overlap=0.5,
        figsize=FIG_SIZES["joyplot"],
        grid=False,
        fill=False,
        background="k",
        linecolor="w",
        linewidth=1,
        legend=False,
        xlabels=False,
        ylabels=False,
        xlim=(vmin, vmax),
    )

    fig.patch.set_facecolor("k")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.show()


def plot_byflavor(
    year_x_day_data: NDArray,
    yr_labels: List[str],
    daily_metrics_df: pd.DataFrame,
    style_key: str = "calm"
) -> None:
    """Plot yearly data as a styled joyplot, highlighting specific years.

    This function takes a 2D array of daily measurements (years x days),
    a list of year labels, and a DataFrame of daily flood metrics. It
    applies a pre-defined style (from the global STYLES dict) to color
    each year's ridgeline according to a flood metric, masks (overpaints)
    missing "zero sentinel" days, and emphasizes flood years via line width.

    Args:
        year_x_day_data (NDArray):
            A 2D NumPy array of shape (n_years, n_days).  Each row is one
            year's daily values; zero indicates a missing data sentinel.
        yr_labels (List[str]):
            A list of length n_years giving a string label for each row in
            year_x_day_data (typically the year as a string).
        daily_metrics_df (DataFrame):
            A pandas DataFrame containing daily-aggregated flood metrics
            (e.g. daily maxima or volumes) indexed by year and day.
        style_key (str, optional):
            Key into the global STYLES mapping, which provides:
              - "years": List of years to highlight
              - "flood_metric": a callable that returns a Series of metrics
              - "norm_func": optional normalization factory
              - "cmap": color map or fixed color
              - other plot styling parameters (linewidths, bgcolor, title).
            Defaults to "calm".

    Returns:
        None: Displays the styled joyplot inline.

    Raises:
        KeyError:
            If `style_key` is not found in the global STYLES dictionary.
        ValueError:
            If `daily_metrics_df` does not contain the necessary index
            or columns expected by the chosen flood_metric.
    """
    style = STYLES[style_key]
    metric = style["flood_metric"](daily_metrics_df)
    # Filtered & ordered” version of metric data
    fv = metric.reindex(map(int, style["years"])).astype(float)
    if style["norm_func"] is not None:
        norm = style["norm_func"](fv.min(skipna=True), fv.max(skipna=True))
        highlight_cols = {
            y: style["default_color"] if np.isnan(fv.loc[int(y)])
            else style["cmap"](norm(fv.loc[int(y)]))
            for y in style["years"]
        }
    else:
        # For styles without normalization, use a single color
        highlight_cols = {y: style["cmap"] for y in style["years"]}

    colors = [highlight_cols.get(lbl, style["default_color"])
              for lbl in yr_labels]
    fig, axes = joyplot(
        year_x_day_data.tolist(), labels=yr_labels,
        xlabels=False, ylabels=False,
        x_range=DAY_RANGE,
        linewidth=style["linewidth"],
        fill=False,
        color=colors,
        kind="values",
        overlap=0.8,
        figsize=FIG_SIZES["joyplot"],
        background=style["bgcolor"]
    )

    for ax, lbl in zip(axes, yr_labels):
        line = ax.get_lines()[0]
        x_data, y_data = line.get_data()
        # /!\ mask those plotted zero flags (NA values)
        zero_points = (y_data == 0)
        ax.plot(
            x_data[zero_points],
            y_data[zero_points],
            color=style["bgcolor"],
            linewidth=style["linewidth"]*1.5,
            solid_capstyle="butt",
            antialiased=False,
            zorder=line.get_zorder()+1
        )
        # highlight if this is a highlighted year
        if str(lbl) in style["years"]:
            line.set_linewidth(style["highlight_width"])
        else:
            line.set_linewidth(style["linewidth"])

        ax.set_xlim(1, 366)
        ax.set_ylim(-500, 8600)
        ax.tick_params(colors="w")
        ax.set_facecolor(style["bgcolor"])

    fig.suptitle(style["title"], color="w", fontsize=14)
    fig.patch.set_facecolor(style["bgcolor"])
    plt.show()


def plot_years(
    year_x_day_data: np.ndarray,
    yr_labels: Sequence[str],
    years: Union[int, range, Sequence[int]],
    interpolate_gaps: bool = True,
    pad: float = 0.05
) -> Tuple[plt.Figure, Sequence[plt.Axes]]:
    """
    For each year in `years`, plot one dark-background subplot of
    daily-max values vs. day-of-year. The vertical limits (ylim)
    are computed *only* from the data of the requested years.

    Args:
      year_x_day_data: np.ndarray with shape (n_years, 366).
      yr_labels: list of str-years corresponding to the rows.
      years: an int, a range, or list of years to plot.
      interpolate_gaps (bool, optional): Fill in missing values gaps on the plots. Defaults to True.
      pad: fractional padding to add above/below data.

    Returns:
      fig, axes
    """
    if not interpolate_gaps:
        year_x_day_data = np.where(
            year_x_day_data == 0, np.nan, year_x_day_data)
    if isinstance(years, int):
        year_list = [years]
    else:
        year_list = list(years)

    year_strs = [str(y) for y in year_list]
    try:
        idx = [yr_labels.index(ys) for ys in year_strs]
    except ValueError:
        missing = set(year_list) - set(map(int, yr_labels))
        raise ValueError(f"Years not found in labels: {sorted(missing)}")

    # Compute y-limits from *only* those rows
    subset = year_x_day_data[idx, :]
    # Compute raw min/max on the subset
    y_min = np.nanmin(subset)
    y_max = np.nanmax(subset)
    y_min = min(y_min, 0)   # Force zero into the bracket
    y_max = max(y_max, 0)
    span = y_max - y_min    # Pad if applicable
    y_limits = (y_min - pad*span, y_max + pad*span)
    n = len(year_list)
    fig, axes = plt.subplots(
        n, 1,
        figsize=(FIG_SIZES["lineplots"][0],
                 FIG_SIZES["lineplots"][1]*n),
        sharex=True
    )
    if n == 1:
        axes = [axes]
    # plot each year
    for ax, row, yr in zip(axes, idx, year_list):
        ax.plot(DAY_RANGE, year_x_day_data[row], color="white", lw=1)
        ax.axhline(0, color="skyblue", ls="--", lw=0.8)
        ax.set_ylim(*y_limits)
        ax.set_facecolor("black")
        ax.set_title(f"Year {yr}", color="white")
        ax.tick_params(colors="white")
        ax.set_xticks(MONTH_TICKS["positions"])
        ax.set_xticklabels(MONTH_TICKS["labels"], color="white")
        for spine in ax.spines.values():
            spine.set_visible(False)
    # shared X-label and layout
    fig.text(0.5, 0.04, "Month", ha="center", color="white")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()


def interpolate_yearly(year_x_day_data: np.ndarray) -> np.ndarray:
    """
    Subfuntion | Given an array (n_years x n_days) with NaNs marking 
    missing days.
    Returns a new array where each row has been linearly
    interpolated along the day-axis.
    """
    # Build a DataFrame so we can use DataFrame.interpolate()
    df = pd.DataFrame(
        year_x_day_data,
        index=[f"yr_{i}" for i in range(year_x_day_data.shape[0])],
        columns=range(year_x_day_data.shape[1])
    )

    # interpolate across columns (i.e. days), limit_direction='both'
    df_interp = df.interpolate(
        method="linear",
        axis=1,
        limit_direction="both"
    )

    return df_interp.to_numpy()


def plot_stable_years(
    year_x_day_data: np.ndarray,
    labels: List[str],
    n_years: int = 3,
    interpolate_gaps: bool = False,
    figsize_per_row: float = 2.0
) -> None:
    """
    Select and plot the N most stable years as a joyplot.

    Given a 2D array (years x days) and a list of year-labels, pick the N
    rows with lowest stddev (ignoring zero sentinels), then optionally
    interpolate those rows, and finally display a joyplot.

    Args:
        year_x_day_data (np.ndarray):
            A 2D array of shape `(n_years, n_days)`, where zeros mark missing
            data points. Each row corresponds to one year, each column to one day.
        labels (List[str]):
            A list of length `n_years` of human-readable labels (e.g. year strings)
            for each row in `year_x_day_data`.
        n_years (int, optional):
            Number of years (rows) to select based on lowest standard deviation.
            Defaults to 3.
        interpolate_gaps (bool, optional):
            If True, perform linear interpolation on NaN gaps *only* in the
            selected subset of years before plotting. Defaults to False.
        figsize_per_row (float, optional):
            Vertical size in inches allocated to each ridgeline. The final
            figure height will be `n_years * figsize_per_row`. Defaults to 2.0.

    Returns:
        None: Displays the plot inline (e.g. in a Jupyter notebook) but does not
        return any object.

    Raises:
        ValueError:
            If `len(labels)` does not match the number of rows in
            `year_x_day_data`.
    """
    # Mask zeros → NaN, for both std‐calculation AND later plotting
    masked_all = np.where(year_x_day_data == 0, np.nan, year_x_day_data)
    # Compute stddev on the raw masked data (no interpolation here!)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        stds = np.nanstd(masked_all, axis=1, ddof=0)
    # Pick the n_years smallest‐std rows
    chosen_idx = np.argsort(stds)[:n_years]
    chosen_idx_sorted = sorted(chosen_idx, key=lambda i: stds[i], reverse=True)
    # Extract just those rows
    data_sel_raw = masked_all[chosen_idx_sorted, :]
    labels_sel = [labels[i] for i in chosen_idx_sorted]
    if interpolate_gaps:
        data_sel = interpolate_yearly(data_sel_raw)
    else:
        data_sel = data_sel_raw

    fig_height = n_years * figsize_per_row
    fig, axes = joyplot(
        data_sel.tolist(),
        labels=labels_sel,
        x_range=DAY_RANGE,
        kind="values",
        linewidth=1.0,
        overlap=0.8,
        fill=False,
        figsize=(10, fig_height),
        background="black"
    )
    fig.subplots_adjust(left=0.05, right=0.98, top=0.98,
                        bottom=0.05, hspace=0.3)

    # Annotate each ridgeline
    days = np.array(list(DAY_RANGE))
    for arr, ax in zip(data_sel, axes):
        arr = np.array(arr, dtype=float)
        if np.all(np.isnan(arr)):
            continue

        # highlight min/max and median
        min_i, max_i = np.nanargmin(arr), np.nanargmax(arr)
        ax.scatter(days[min_i], arr[min_i], color="orange", s=30)
        ax.scatter(days[max_i], arr[max_i], color="cyan",   s=30)
        ax.text(days[min_i], arr[min_i], f"{int(arr[min_i])} mm",
                color="orange", ha="right", va="top",    fontsize=8)
        ax.text(days[max_i], arr[max_i], f"{int(arr[max_i])} mm",
                color="cyan",   ha="left",  va="bottom", fontsize=8)
        med = np.nanmedian(arr)
        ax.hlines(med, days[0], days[-1],
                  colors="yellow", linestyles=":", linewidth=0.8, alpha=0.7)
        ax.text(days[-1] + 2, med, f"med={med:.1f} mm",
                color="yellow", va="center", fontsize=7)
        ax.set_xlim(days[0], days[-1])
        ax.set_xlabel("Day of Year")
        ax.set_ylabel("water_level_mm")
        ax.grid(axis="x", linestyle="--", alpha=0.7)

    plt.show()


def plot_diverging_years(
    year_x_day_data: np.ndarray,
    yr_labels: List[str],
    years: List[int],
    interpolate_gaps: bool = True,
    y_limits: tuple[int, int] = (-1000, 9000)
) -> None:
    """
    Plot line chart of daily-max values for specified years.

    Args:
        year_x_day_data (np.ndarray): shape (n_years, 366), with NaNs for missing days
        yr_labels (List[str]): list of years (as strings) mapping to rows of data
        years (List[int]): which years to plot
        interpolate_gaps (bool): if True, linearly interpolate NaN gaps
        y_limits (tuple): (ymin, ymax) for all subplots
    """
    # map year → row index
    year_to_idx = {int(y): i for i, y in enumerate(yr_labels)}
    idxs = [year_to_idx[y] for y in years]
    sel = year_x_day_data[idxs, :].astype(float)

    if interpolate_gaps:    # Fill all NaNs
        sel = interpolate_yearly(sel)
    else:
        # break the line on zeros
        sel = np.where(sel == 0, np.nan, sel)

    fig, axes = plt.subplots(
        nrows=len(years),
        ncols=1,
        figsize=(
            FIG_SIZES["lineplots"][0],
            FIG_SIZES["lineplots"][1] * len(years),
        ),
        sharex=True
    )
    if len(years) == 1:
        axes = [axes]
    for ax, yr, row in zip(axes, years, sel):
        ax.plot(DAY_RANGE, row, color="white", lw=1)
        ax.axhline(0, color="skyblue", ls="--", lw=0.8)
        ax.set_ylim(*y_limits)
        ax.set_facecolor("black")
        ax.set_title(f"Year {yr}", color="white")
        ax.tick_params(colors="white")
        ax.set_xticks(MONTH_TICKS["positions"])
        ax.set_xticklabels(MONTH_TICKS["labels"], color="white")
        for spine in ax.spines.values():
            spine.set_visible(False)
    fig.text(0.5, 0.04, "Month", ha="center", color="white")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()